# Ayan's `XGBDT SHAP` Values Analysis

## Purpose: 

{TODO}

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

import glob, json, os

import matplotlib.pyplot as plt 
import seaborn as sns 


In [2]:
ayan_shap_folder = "/project/PlatigLab/data/collaborators/BWH/3_XGBDT_SHAP_data_2024_07/"
cell_lines = ["K562", "HepG2"]
distance_thresholds = list({file.split("/")[-1].split("-")[1] for file in glob.glob(f"{ayan_shap_folder}/**/*data.dat", recursive=True)})
best_performing_distance_threshold=100

shap_files = sorted(glob.glob(f"{ayan_shap_folder}/**/*data.dat", recursive=True))

actual_psi_column_name = "target"
predicted_psi_column_name = "psi_hat"

# dictionary to rename the splice junction positions to numbers
splice_junction_position_renaming = {
    "5_left": 1, 
    "5_right": 2, 
    "center_left": 3, 
    "center_right": 4, 
    "3_left": 5, 
    "3_right": 6
}

rbp_comparisons_file = "../outputs/rbp_comparisons/rbp_comparisons.json"

In [3]:
rbp_ppi=pd.read_excel("../../../inputs/RBP-RBP_PPI/lang_et_al_rec-y2h_screening_results.xlsx")
rbp_ppi = rbp_ppi[(rbp_ppi["sumIS"]>=7.1)]

for column in ["Protein A", "Protein B", "UniProt accessions A", "UniProt accessions B"]: 
    rbp_ppi[column] = rbp_ppi[column].str.lower()

rbp_ppi.shape
rbp_ppi.head()



(2416, 57)

,Protein A,Protein B,UniProt species A,UniProt species B,UniProt accessions A,UniProt accessions B,UniProt IDs A,UniProt IDs B,Times detected (RIS > 0),avgIS,sumIS,Found in both orientations,H47_avgIS,H47_sumIS,H47_found in both orientations,H47_times detected (RIS > 0),"Biogrid_all direct evidence (Y2H, reconstituted complex, structure)",Biogrid_all,Hippie,HuRI,Known interaction,Protein A has known interactions,Protein B has known interactions,HPA nuclear A,HPA nuclear B,HPA cytoplasmic A,HPA cytoplasmic B,HPA main locations A,HPA additional locations A,HPA main locations B,HPA additional locations B,HPA shared locations,Youn et al. 2018 (PMID 29395067),NanoBRET,NanoBRET MBU,NanoBRET MBU std,ENCODE eCLIP data A,ENCODE eCLIP data B,ENCODE eCLIP binding sites A,ENCODE eCLIP binding sites B,Jaccard index,Cobinding prob p(A|B),Cobinding prob p(B|A),Cobinding prob p(A|B) (≤54 nt),Cobinding prob p(B|A) (≤54 nt),Close binding events (≤54 nt) A vs. B,Fraction of close binding events A vs. B,Fraction of close binding events in random data A vs. B,Ratio of fractions A vs. B,Resampling p-value A vs. B,Resampling Wilcoxon p-value A vs. B,Close binding events (≤54 nt) B vs. A,Fraction of close binding events B vs. A,Fraction of close binding events in random data B vs. A,Ratio of fractions B vs. A,Resampling p-value B vs. A,Resampling Wilcoxon p-value B vs. A
0,ctbp1,rbm14,HUMAN,MOUSE,q13363,q8c2q3,CTBP1_HUMAN,RBM14_MOUSE,10,8.41,17.19,1,NaN,NaN,NaN,NaN,1,1,1,0,1,1,1,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nuclear speckles,NaN,NaN,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rbm14,cpsf6,MOUSE,MOUSE,q8c2q3,q6nvf9,RBM14_MOUSE,CPSF6_MOUSE,11,13.98,16.03,0,NaN,NaN,NaN,NaN,0,0,0,0,0,1,0,1.0,1.0,0.0,0.0,Nuclear speckles,NaN,Nuclear speckles;Nucleoplasm,NaN,Nuclear speckles,0,NaN,NaN,NaN,0,1,0,791,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ptbp1,ptbp2,HUMAN,HUMAN,p26599,q9uka9,PTBP1_HUMAN,PTBP2_HUMAN,8,4.49,15.73,1,0.00,5.47,0.0,5.0,0,0,0,0,0,0,0,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nucleoplasm,NaN,Nucleoplasm,0,pos,13.34,0.94,1,0,16175,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sf1,ewsr1,HUMAN,MOUSE,q15637,q61545,SF01_HUMAN,EWS_MOUSE,3,3.29,15.50,0,7.17,11.66,0.0,3.0,1,1,1,0,1,1,1,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nucleoplasm,Nucleoli,Nucleoplasm,0,NaN,NaN,NaN,0,1,0,8788,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,srsf11,srpk2,HUMAN,HUMAN,q05519,p78362,SRS11_HUMAN,SRPK2_HUMAN,10,9.52,14.79,1,2.32,5.06,0.0,5.0,0,1,1,1,1,1,1,1.0,1.0,0.0,1.0,Nuclear speckles,NaN,Cytosol;Nucleoplasm,NaN,NaN,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
uniprot_id_mapping = pd.read_csv("../../../inputs/RBP-RBP_PPI/uniprot_mapping.tsv", sep="\t")

for column in uniprot_id_mapping: 
    uniprot_id_mapping[column] = uniprot_id_mapping[column].str.lower()

uniprot_id_mapping.head()

,Gene stable ID,Gene name,Gene Synonym,UniProtKB Gene Name ID
0,ensg00000286112,kyat1,ccbl1,a0a494c066
1,ensg00000171097,kyat1,ccbl1,q16773
2,ensg00000171097,kyat1,ccbl1,q5t276
3,ensg00000171097,kyat1,ccbl1,q5t277
4,ensg00000171097,kyat1,ccbl1,q5t278


## Despite larger distance thresholds having lower `R^2` performance, do they do better at predicting non-0/1 `PSI`? 

In [5]:

# for cell_line in cell_lines: 
    
#     for threshold in distance_thresholds: 
#         f"{cell_line} {threshold}"
        
#         files = [ file for file in shap_files if f"{cell_line}-{threshold}-" in file and "-train-" not in file ]
#         assert len(files)==2, [file.split("/")[-1] for file in files]        
        

#         plt.figure(dpi=100, figsize=(10,10))
        
        
#         df = [pd.read_csv(file, sep=",", usecols=[actual_psi_column_name, predicted_psi_column_name]) for file in files if "-train-" not in file]
        
#         df = pd.concat(df)
        
# #         sns.scatterplot(
# #             df, 
# #             x="target", 
# #             y="psi_hat", 
# #             size=0.00000000000001
        
# #         )

#         plt.hist2d(
        
#             df["target"].to_list(), 
#             df["psi_hat"].to_list(), 
#             bins=10
            
#         )
          
#         plt.show()

In [6]:
data_dict = {}

for cell_line in cell_lines: 
    data_dict[cell_line] = {}
    
    f"{cell_line}"

    files = [ file for file in shap_files if f"{cell_line}-{best_performing_distance_threshold}-" in file and "-train-" not in file ]
    assert len(files)==2, [file.split("/")[-1] for file in files]    

    df = [ pd.read_csv(file, sep=",", nrows=0) for file in files ]

    data_dict[cell_line] = pd.concat(df)

'K562'

'HepG2'

In [7]:
for cell_line in data_dict: 
    data_dict[cell_line].shape

(0, 1670)

(0, 1262)

## Do `RBP PPI pairs` have similar `SHAP` in places where they bind together vs. don’t bind?

In [8]:
def retrieve_rbp_by_synonym(missing_rbp, missing_rbp_id, shap_rbp_columns, uniprot_mapping): 

    rbp_name = []
    
    mappings = pd.concat(
        [
            uniprot_mapping[uniprot_mapping["UniProtKB Gene Name ID"]==missing_rbp_id], 
            uniprot_mapping[uniprot_mapping["Gene name"]==missing_rbp], 
            uniprot_mapping[uniprot_mapping["Gene Synonym"]==missing_rbp]
        ]
    ).drop_duplicates()
        

    if len(mappings)==0: 
        return None 
                
    else: 
        rbp_synonyms = mappings["Gene name"].to_list() # + mappings["Gene Synonym"].to_list()

        for rbp in rbp_synonyms: 

            if rbp in shap_rbp_columns: 
                rbp_name.append(rbp)

    if len(rbp_name)==0: 
        return None 

    else: 
        rbp_name = list(set(rbp_name))
        assert len(rbp_name)==1
        return rbp_name[0]
        


In [9]:
def retrieve_rbp_comparison_cols(df, rbp_ppi, uniprot_mapping): 

    interacting_columns = set()
    
    shap_rbp_columns = list({ column.split("_")[0].lower() for column in df.columns.to_list() if "_right" in column or "_left" in column })
        
    for index, row in rbp_ppi.iterrows(): 

        if row["Protein A"].lower() in shap_rbp_columns: 
            rbp_1 = row["Protein A"].lower()
        else: 
            rbp_1 = retrieve_rbp_by_synonym(row["Protein A"].lower(), row["UniProt accessions A"].lower(), shap_rbp_columns, uniprot_mapping) 
            
        if row["Protein B"].lower() in shap_rbp_columns: 
            rbp_2 = row["Protein B"].lower()
        else: 
            rbp_2 = retrieve_rbp_by_synonym(row["Protein B"].lower(), row["UniProt accessions B"].lower(), shap_rbp_columns, uniprot_mapping) 

        if rbp_1!=None and rbp_2!=None: 
            interacting_columns.add((rbp_1, rbp_2))

    return interacting_columns

In [10]:

if os.path.exists(rbp_comparisons_file):
    with open(rbp_comparisons_file) as file: 
        comparison_rbps = json.load(file)

else: 

    comparison_rbps = {}
    
    for cell_line in data_dict: 
        comparison_rbps[cell_line] = sorted(retrieve_rbp_comparison_cols(data_dict[cell_line], rbp_ppi, uniprot_id_mapping))

    with open(rbp_comparisons_file, 'w') as out_file: 
        json.dump(comparison_rbps, out_file, indent=4)

In [11]:
print(comparison_rbps)

{'K562': [['cpeb4', 'cstf2t'], ['cstf2t', 'ewsr1'], ['ddx21', 'rbfox2'], ['fmr1', 'fxr2'], ['fmr1', 'rybp'], ['fxr1', 'fxr2'], ['gtf2f1', 'pcbp1'], ['hnrnpk', 'apobec3c'], ['hnrnpk', 'ewsr1'], ['hnrnpk', 'hnrnpu'], ['hnrnpk', 'larp4'], ['hnrnpk', 'qki'], ['hnrnpk', 'srsf7'], ['ilf3', 'cpsf6'], ['mbnl1', 'qki'], ['nono', 'cpsf6'], ['nono', 'cstf2t'], ['nono', 'ewsr1'], ['pcbp1', 'apobec3c'], ['pcbp1', 'hnrnpk'], ['pcbp1', 'igf2bp2'], ['pcbp1', 'rbfox2'], ['ppil4', 'rbfox2'], ['ptbp1', 'igf2bp1'], ['ptbp1', 'igf2bp2'], ['ptbp1', 'pcbp1'], ['ptbp1', 'qki'], ['ptbp1', 'tardbp'], ['pum2', 'qki'], ['qki', 'pcbp1'], ['rbfox2', 'ddx42'], ['rbfox2', 'igf2bp2'], ['rbfox2', 'mbnl1'], ['rbfox2', 'pum2'], ['rbfox2', 'qki'], ['rbm15', 'cpsf6'], ['rbm15', 'ewsr1'], ['rbm15', 'srsf9'], ['sf3b4', 'rbfox2'], ['sf3b4', 'taf15'], ['srsf9', 'hnrnpa1'], ['supv3l1', 'rbfox2'], ['taf15', 'cpsf6'], ['taf15', 'cstf2t'], ['taf15', 'hnrnpk'], ['taf15', 'srsf1'], ['taf15', 'srsf9'], ['tia1', 'rbm22'], ['tra2a', 's

In [ ]:
for cell_line in data_dict: 
    
    tmp_df = data_dict[cell_line] 
    
    for rbp_pair in comparison_rbps[cell_line]: 

        for position in splice_junction_position_renaming: 

            use_cols = []

            for column in tmp_df.columns.to_list(): 
                
                if position in "_".join(column.split("_")[1:]): 

                    if column.split("_")[0].lower() == rbp_pair[0] or column.split("_")[0].lower() == rbp_pair[1]: 
                        use_cols.append(column)

            use_cols = use_cols + [actual_psi_column_name, predicted_psi_column_name]

            assert len(use_cols)==6

            files = [ file for file in shap_files if f"{cell_line}-{best_performing_distance_threshold}-" in file and "-train-" not in file ]
            assert len(files)==2, [file.split("/")[-1] for file in files]    
        
            data_df = [ pd.read_csv(file, sep=",", usecols=use_cols) for file in files ]
            data_df = pd.concat(data_df)

            binding_cols = sorted([column for column in data_df.columns.to_list() if column.endswith(position)])
            shap_cols = sorted([column for column in data_df.columns.to_list() if column.endswith("shap")])
            assert len(binding_cols)==2 and len(shap_cols)==2

            double_binding = ((data_df[binding_cols[0]]==1) & (data_df[binding_cols[1]]==1))
            
            double_binding = data_df.loc[double_binding].copy(deep=True)
            all_others = data_df.loc[~data_df.index.isin(double_binding.index)].copy(deep=True)

            if len(double_binding)!=0: 
                
                double_binding["shap_difference"] = double_binding[shap_cols[0]] - double_binding[shap_cols[1]]
                double_binding["Binding Mode"] = "Both Binding"
                
                all_others["shap_difference"] = all_others[shap_cols[0]] - all_others[shap_cols[1]]
                all_others["Binding Mode"] = "All Others"
                
                data_df = pd.concat(
                    [
                        double_binding[["shap_difference", "Binding Mode"]], 
                        all_others[["shap_difference", "Binding Mode"]]
                    ]
                )
    
                assert all(abs(data_df["shap_difference"]) < 1)
                
                plt.figure(dpi=100)
    
                plt.title("{}: {}-{} {}".format(cell_line, rbp_pair[0], rbp_pair[1], position))
                sns.boxplot(
                    data_df, 
                    x="Binding Mode",
                    y="shap_difference"
                )
    
                plt.ylim(-1,1)

                plt.text(-0.25, 0.75, "Both Binding #: \n{}".format(len(double_binding)))
                plt.text(0.75, 0.75, "All Others #: \n{}".format(len(all_others)))
    
                plt.savefig(
                    "../plots/{}_{}-{}_{}".format(cell_line, rbp_pair[0], rbp_pair[1], position), 
                    dpi=100, 
                    bbox_inches="tight"
                )
    
                plt.show()  